In [2]:
"""
Apply every merge decision from tier 2 and tier 3 to expanded_triples.xlsx,
producing the final resolved entity columns. This is the missing link
between all the review sheets and an actually-usable resolved graph.

Sources combined into one lookup:
  - tier2_llm_code_extraction_review.xlsx, sheet "proposed_auto_merges":
    confident code-based merges, no manual review needed.
  - tier3_llm_adjudication_review.xlsx, sheet "adjudication_results":
    ONLY rows where is_same_entity = True AND needs_manual_review = False.
    Rows flagged needs_manual_review = True are NOT applied here, they
    still need your decision, applying them automatically would defeat
    the whole point of that flag.

Both sources map (doi, raw_entity_name) -> canonical_name. Applied to
expanded_source/expanded_target to produce final_source/final_target.

Nothing here is silent: a resolved_by column records which stage (or
none) resolved each row, and rows needing your attention are written
to a separate sheet rather than skipped without a trace.

Designed for Jupyter/Colab execution. No __main__ guard.
"""

import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "expanded_triples.xlsx"
SOURCE_COL = "expanded_source"
TARGET_COL = "expanded_target"
DOI_COL = "doi"

TIER2_PATH = "tier2_llm_code_extraction_review.xlsx"
TIER2_SHEET = "proposed_auto_merges"

TIER3_PATH = "tier3_llm_adjudication_review.xlsx"
TIER3_SHEET = "adjudication_results"

OUTPUT_TRIPLES_XLSX = "final_resolved_triples.xlsx"
OUTPUT_AUDIT_XLSX = "entity_merge_audit.xlsx"

# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)
print(f"Loaded {len(df)} triples")

tier2_df = pd.read_excel(TIER2_PATH, sheet_name=TIER2_SHEET)
tier3_df = pd.read_excel(TIER3_PATH, sheet_name=TIER3_SHEET)

# ---------------------------------------------------------------
# BUILD THE COMBINED LOOKUP: (doi, raw_name) -> canonical_name
# ---------------------------------------------------------------
lookup = {}
lookup_source = {}  # (doi, raw_name) -> which stage resolved it, for the audit trail

for _, row in tier2_df.iterrows():
    key = (row["doi"], str(row["raw_name"]).strip())
    lookup[key] = row["proposed_canonical_name"]
    lookup_source[key] = "tier2_auto_merge"

# Only apply tier 3 rows the LLM AND the review flag both agree on:
# is_same_entity = True AND needs_manual_review = False. No hand-written
# exception list to maintain, this scales to however many rows there
# are. If you want to override a specific row's decision, edit
# needs_manual_review (or is_same_entity) directly in
# tier3_llm_adjudication_review.xlsx and rerun, rather than tracking a
# separate list here.
n_tier3_applied = 0
n_tier3_excluded = 0
for _, row in tier3_df.iterrows():
    if row["is_same_entity"] is not True:
        continue
    if bool(row.get("needs_manual_review", False)):
        n_tier3_excluded += 1
        continue

    canonical = row["proposed_canonical_name"]
    for m in str(row["members"]).split("; "):
        key = (row["doi"], m.strip())
        lookup[key] = canonical
        lookup_source[key] = "tier3_adjudication"
        n_tier3_applied += 1

print(f"Tier 2 confident merges loaded: {len(tier2_df)}")
print(f"Tier 3 merges applied: {n_tier3_applied}")
print(f"Tier 3 merges excluded (needs_manual_review = True): {n_tier3_excluded}")
print(f"Total lookup entries: {len(lookup)}")

# ---------------------------------------------------------------
# APPLY TO THE TRIPLES
# ---------------------------------------------------------------
def resolve(entity, doi):
    key = (doi, str(entity).strip())
    if key in lookup:
        return lookup[key], lookup_source[key]
    return entity, None


final_sources, final_targets = [], []
source_resolved_by, target_resolved_by = [], []

for _, row in df.iterrows():
    fs, fs_by = resolve(row[SOURCE_COL], row[DOI_COL])
    ft, ft_by = resolve(row[TARGET_COL], row[DOI_COL])
    final_sources.append(fs)
    final_targets.append(ft)
    source_resolved_by.append(fs_by)
    target_resolved_by.append(ft_by)

df["final_source"] = final_sources
df["final_target"] = final_targets
df["final_source_resolved_by"] = source_resolved_by
df["final_target_resolved_by"] = target_resolved_by

n_source_merged = df["final_source_resolved_by"].notna().sum()
n_target_merged = df["final_target_resolved_by"].notna().sum()
print(f"\nRows with source merged: {n_source_merged}")
print(f"Rows with target merged: {n_target_merged}")

# ---------------------------------------------------------------
# NODE COUNT BEFORE/AFTER, the actual "did this help" number
# ---------------------------------------------------------------
before_nodes = pd.concat([df[SOURCE_COL], df[TARGET_COL]]).astype(str).str.strip().nunique()
after_nodes = pd.concat([df["final_source"], df["final_target"]]).astype(str).str.strip().nunique()
print(f"\nUnique nodes before entity merge: {before_nodes}")
print(f"Unique nodes after entity merge: {after_nodes}")
print(f"Nodes collapsed: {before_nodes - after_nodes}")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
df.to_excel(OUTPUT_TRIPLES_XLSX, index=False)
print(f"\nSaved final resolved triples to {OUTPUT_TRIPLES_XLSX}")

excluded_df = tier3_df[
    (tier3_df["is_same_entity"] == True) & (tier3_df["needs_manual_review"] == True)  # noqa: E712
]

readme_rows = [
    "HOW TO READ THIS FILE",
    "",
    "- merge_lookup: every (doi, raw_name) -> canonical_name pair actually applied, with which",
    "  stage resolved it (tier2_auto_merge or tier3_adjudication).",
    "",
    "- excluded_needs_review: tier 3 rows the LLM said were the same entity, but needs_manual_review",
    "  was True, so they were NOT applied. To include one, edit needs_manual_review to False (or",
    "  is_same_entity as needed) directly in tier3_llm_adjudication_review.xlsx and rerun this script.",
    "",
    f"Nodes before merge: {before_nodes}. Nodes after: {after_nodes}. Collapsed: {before_nodes - after_nodes}.",
]
readme_df = pd.DataFrame({"": readme_rows})

merge_lookup_df = pd.DataFrame([
    {"doi": k[0], "raw_name": k[1], "canonical_name": v, "resolved_by": lookup_source[k]}
    for k, v in lookup.items()
])

with pd.ExcelWriter(OUTPUT_AUDIT_XLSX) as writer:
    readme_df.to_excel(writer, sheet_name="READ_ME_FIRST", index=False)
    merge_lookup_df.to_excel(writer, sheet_name="merge_lookup", index=False)
    excluded_df.to_excel(writer, sheet_name="excluded_needs_review", index=False)

print(f"Saved audit workbook to {OUTPUT_AUDIT_XLSX}")

Loaded 10324 triples
Tier 2 confident merges loaded: 0
Tier 3 merges applied: 0
Tier 3 merges excluded (needs_manual_review = True): 0
Total lookup entries: 0

Rows with source merged: 0
Rows with target merged: 0

Unique nodes before entity merge: 3100
Unique nodes after entity merge: 3100
Nodes collapsed: 0

Saved final resolved triples to final_resolved_triples.xlsx
Saved audit workbook to entity_merge_audit.xlsx
